Desigualdades en el acceso a hospitalización compleja:

● ¿Existen diferencias regionales en el acceso a hospitalizaciones de alta complejidad en
el sistema público de salud?

● ¿La distribución de pacientes de alta complejidad se concentra en ciertos hospitales?

Vamos a filtrar todos los hospitales con mortalidad hospitalaria alta. 
Luego chequear Hospitales con alta frecuencia de traslado y revisar donde están siendo trasladados. 
Para así entender que hospital está generando mas traslados y entender que esta sucediendo en ese hospital en particular.


# ¿Cuál es la relación entre los altos índices de mortalidad hospitalaria y los patrones de derivación de pacientes en la red pública de salud, y qué características clínicas o de gestión explican el comportamiento de los hospitales con mayor emisión de traslados?

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob as gb
import os
import csv

In [7]:
# Definimos las columnas que realmente nos interesan para evitar cargar datos innecesarios
columnas_necesarias = [
    'CIP_ENCRIPTADO', 'COD_HOSPITAL', 'COMUNA', 'PROVINCIA', 'TIPO_PROCEDENCIA', 'FECHA_INGRESO', 'FECHAALTA', 'TIPOALTA', 'DIAGNOSTICO1', 'PROCEDIMIENTO1', 'USOSPABELLON', 'IR_29301_PESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD', 'HOSPPROCEDENCIA'
]

# 1. Buscamos todos los archivos .csv dentro de la carpeta llamada 'csv'
ruta_archivos = 'csv/*.csv'
lista_archivos = gb.glob(ruta_archivos)

# Lista vacía donde guardaremos cada DataFrame temporalmente
dataframes = []

print(f"Se encontraron {len(lista_archivos)} archivos. Comenzando la lectura...\n")

# El orden del algoritmo es importante, algunos archivos pueden ser leidos con UTF-16, otros con UTF-8-SIG, y algunos podrían requerir Latin-1.
# Si leemos los archivos del 2022 y 2023 con latin-1 se devuelve un df lleno de NaN y columnas con nombres desconocidas
# 2024 TIENE que ser leído con latin-1 para que se reconozcan las columnas correctamente, pero el resto de los archivos pueden ser leidos con UTF-16 o UTF-8-SIG

# 2. Iteramos sobre cada archivo encontrado
for archivo in lista_archivos:
    print(f"Procesando: {archivo}...")
    
    # Usamos la configuración robusta que funcionó
    try:
        df_temp = pd.read_csv(
            archivo, 
            sep='|', 
            decimal=',',
            encoding='utf-16',          
            quoting=csv.QUOTE_NONE,
            usecols=columnas_necesarias,  
            index_col=False,            
            on_bad_lines='warn',        
            low_memory=False
        )
    except UnicodeError:
        try:
            # Plan B por si algún archivo en la carpeta tiene codificación distinta
            df_temp = pd.read_csv(
                archivo, 
                sep='|', 
                decimal=',',
                encoding='utf-8-sig',       
                quoting=csv.QUOTE_NONE,
                usecols=columnas_necesarias, 
                index_col=False,
                on_bad_lines='warn',
                low_memory=False
            )
        except UnicodeDecodeError:
            #Plan C por si ninguna codificación funciona, intentamos con 'latin-1' 
            df_temp = pd.read_csv(
                archivo, 
                sep='|', 
                decimal=',',
                encoding='latin-1',       
                quoting=csv.QUOTE_NONE,
                usecols=columnas_necesarias, 
                index_col=False,
                on_bad_lines='warn',
                low_memory=False
            )
    
    # Agregamos el DataFrame a nuestra lista
    dataframes.append(df_temp)

# 3. Concatenamos todo de una sola vez
# ignore_index=True es clave: resetea el conteo de filas (0, 1, 2...) para que no se repitan
df_consolidado = pd.concat(dataframes, ignore_index=True)

print("\n¡Proceso terminado!")
print(f"El DataFrame final consolidado tiene {df_consolidado.shape[0]} filas y {df_consolidado.shape[1]} columnas.")

Se encontraron 6 archivos. Comenzando la lectura...

Procesando: csv\GRD_PUBLICO_2019.csv...
Procesando: csv\GRD_PUBLICO_2020.csv...
Procesando: csv\GRD_PUBLICO_2021.csv...
Procesando: csv\GRD_PUBLICO_2022.csv...
Procesando: csv\GRD_PUBLICO_2023.csv...
Procesando: csv\GRD_PUBLICO_2024.csv...

¡Proceso terminado!
El DataFrame final consolidado tiene 5808536 filas y 15 columnas.


8                     HOSPITAL CLORINDA AVELLO (SANTA JUANA)
92                           HOSPITAL SAN AGUSTÍN DE FLORIDA
93              HOSPITAL INTERCULTURAL KALLVULLANKA (CAÑETE)
99                                          HOSPITAL DE LOTA
149                       HOSPITAL SANTA FILOMENA (GRANEROS)
                                 ...                        
5808459                           HOSPITAL SAN JOS� (PARRAL)
5808479    HOSPITAL PROVINCIAL DEL HUASCO MONSE�OR FERNAN...
5808499                               HOSPITAL DE SAN CARLOS
5808511           HOSPITAL DR. ANTONIO TIRADO LANAS (OVALLE)
5808529                          HOSPITAL SAN JOS� (CORONEL)
Name: HOSPPROCEDENCIA, Length: 703854, dtype: object